# AI Resume Screening - EDA


## Objectives :
### - Understand dataset structure and quality
### - Identify missing values and anomalies
### - Discover patterns and relationships
### - Prepare data for downstream modeling

## Below are the grouping of columns under different lables

| Type                      | Columns                                                          |
| ------------------------- | ---------------------------------------------------------------- |
|    Candidate Info         | `candidate_id`, `primary_role`, `primary_domain`                 |
|    Experience             | `years_experience`                                               |
|    Education              | `highest_education`, `education_field`, `institution_tier`       |
|    Skills                 | `technical_skills_raw`, `tools_platforms_raw`, `soft_skills_raw` |
|    NLP Text               | `experience_summary`, `project_summary`                          |
|    Flags (VERY IMPORTANT) | `ml_experience_flag`, `cloud_experience_flag`, etc.              |
|    Scores                 | `keyword_density_score`, `profile_completeness_score`            |


##  1. Import Libraries & Setup

In [367]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV


## 2.Load Data


In [368]:
 
df = pd.read_csv("../data/parsed_resumes.csv")

print(df.shape)
print(df.info())
print(df.describe())

(5000, 34)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 34 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   candidate_id                            5000 non-null   str    
 1   resume_file_name                        5000 non-null   str    
 2   candidate_name                          5000 non-null   str    
 3   primary_domain                          5000 non-null   str    
 4   primary_role                            5000 non-null   str    
 5   years_experience                        5000 non-null   int64  
 6   highest_education                       4730 non-null   str    
 7   education_field                         4763 non-null   str    
 8   institution_tier                        5000 non-null   str    
 9   technical_skills_raw                    5000 non-null   str    
 10  tools_platforms_raw                     5000 non-null   str 

In [369]:
df.head(50)

,candidate_id,resume_file_name,candidate_name,primary_domain,primary_role,years_experience,highest_education,education_field,institution_tier,technical_skills_raw,...,enterprise_systems_experience_flag,offshore_onsite_model_experience_flag,multi_vendor_coordination_flag,process_compliance_experience_flag,documentation_heavy_role_flag,mentoring_experience_flag,stakeholder_management_experience_flag,resume_length_words,keyword_density_score,profile_completeness_score
0,CND_100001,DataAnalyst_1.pdf,Candidate_1,Data Science,Data Analyst,7,LLM,Mathematics,Tier-1,"Machine Learning, Power BI, Pandas, Python",...,1,1,1,1,0,1,1,56,0.74,0.61
1,CND_100002,profile_2.pdf,Candidate_2,IT,Senior Software Engineer,10,Masters,Computer Science,Tier-2,"Kubernetes, Git, Docker, Python",...,1,1,1,1,1,1,1,54,0.89,0.89
2,CND_100003,Legal_candidate_3.pdf,Candidate_3,Legal,Compliance Officer,7,LLM,Law,Tier-1,"Due Diligence, Legal Research, Compliance, Con...",...,1,1,1,1,1,1,1,57,0.40,0.56
3,CND_100004,IT_candidate_4.pdf,Candidate_4,IT,Senior Software Engineer,9,LLM,Computer Science,Tier-1,"Kubernetes, AWS, SQL, Docker",...,1,1,1,1,1,1,1,55,0.74,0.89
4,CND_100005,IT_candidate_5.pdf,Candidate_5,IT,Software Engineer,9,Masters,Information Technology,Tier-1,"Java, Kubernetes, Git, AWS",...,1,1,1,1,1,1,1,51,0.58,0.59
5,CND_100006,Data Science_candidate_6.pdf,Candidate_6,Data Science,Junior Data Scientist,2,Bachelors,Mathematics,Tier-1,"NLP, Machine Learning, SQL, Python",...,0,0,0,0,0,0,0,54,0.68,0.93
6,CND_100007,DevOpsEngineer_7.pdf,Candidate_7,IT,DevOps Engineer,9,NaN,Computer Science,Tier-1,"Kubernetes, AWS, SQL, Java",...,1,1,1,1,1,1,1,55,0.81,0.58
7,CND_100008,resume_8_Full_Stack_Developer.pdf,Candidate_8,IT,Full Stack Developer,10,Bachelors,Information Technology,Tier-1,"Kubernetes, AWS, Java, Git",...,1,1,1,1,1,1,1,51,0.41,0.63
8,CND_100009,resume_9_Cloud_Engineer.pdf,Candidate_9,IT,Cloud Engineer,5,Masters,Information Technology,Tier-2,"AWS, Kubernetes, Java, Ptyhon",...,0,1,0,1,1,0,1,52,0.44,0.83
9,CND_100010,resume_10_ML_Engineer.pdf,Candidate_10,Data Science,ML Engineer,5,Masters,Mathematics,Tier-3,"Machine Learning, NLP, Power BI, Pythno",...,0,1,0,1,0,0,1,56,0.28,0.58


In [370]:
df.columns

Index(['candidate_id', 'resume_file_name', 'candidate_name', 'primary_domain',
       'primary_role', 'years_experience', 'highest_education',
       'education_field', 'institution_tier', 'technical_skills_raw',
       'tools_platforms_raw', 'soft_skills_raw', 'experience_summary',
       'project_summary', 'key_achievements', 'management_experience_flag',
       'people_management_flag', 'project_management_experience_flag',
       'agile_scrum_experience_flag', 'client_facing_experience_flag',
       'delivery_lead_experience_flag', 'cloud_experience_flag',
       'ml_experience_flag', 'compliance_experience_flag',
       'enterprise_systems_experience_flag',
       'offshore_onsite_model_experience_flag',
       'multi_vendor_coordination_flag', 'process_compliance_experience_flag',
       'documentation_heavy_role_flag', 'mentoring_experience_flag',
       'stakeholder_management_experience_flag', 'resume_length_words',
       'keyword_density_score', 'profile_completeness_score']

In [371]:
df.columns.value_counts()

candidate_id                              1
resume_file_name                          1
candidate_name                            1
primary_domain                            1
primary_role                              1
years_experience                          1
highest_education                         1
education_field                           1
institution_tier                          1
technical_skills_raw                      1
tools_platforms_raw                       1
soft_skills_raw                           1
experience_summary                        1
project_summary                           1
key_achievements                          1
management_experience_flag                1
people_management_flag                    1
project_management_experience_flag        1
agile_scrum_experience_flag               1
client_facing_experience_flag             1
delivery_lead_experience_flag             1
cloud_experience_flag                     1
ml_experience_flag              

## Right now our columns are raw.
## We need to organize them into logical feature groups → this helps in:

   ### 1 EDA
   ### 2 Feature engineering
   ### 3 Final report

In [372]:
## Group the features : 

# Candidate Identity-->'candidate_id','resume_file_name','candidate_name'
# Core Profile--> 'primary_domain','primary_role','years_experience'--->Used for:Domain match,Role match,Experience scoring
# Education--> 'highest_education','education_field','institution_tier'
# Skills--->'technical_skills_raw','tools_platforms_raw','soft_skills_raw'
# Text Data--->'experience_summary','project_summary','key_achievements'(NLP)
#flag-->col for col in df.columns if 'flag' in col
#Scoring Metrics-->'resume_length_words','keyword_density_score','profile_completeness_score'--> used for Resume quality scoring


In [373]:
id_cols = [
    'candidate_id',
    'resume_file_name',
    'candidate_name'
]

In [374]:
profile_cols = [
    'primary_domain',
    'primary_role',
    'years_experience'
]

In [375]:
education_cols = [
    'highest_education',
    'education_field',
    'institution_tier'
]

In [376]:
skills_cols = [
    'technical_skills_raw',
    'tools_platforms_raw',
    'soft_skills_raw'
]

In [377]:
text_cols = [
    'experience_summary',
    'project_summary',
    'key_achievements'
]

In [378]:
flag_cols = [
    col for col in df.columns if 'flag' in col
]

In [379]:
score_cols = [
    'resume_length_words',
    'keyword_density_score',
    'profile_completeness_score'
]

# Combine All to Create a dictionary:


In [380]:

column_groups = {
    "id": id_cols,
    "profile": profile_cols,
    "education": education_cols,
    "skills": skills_cols,
    "text": text_cols,
    "flags": flag_cols,
    "scores": score_cols
}

# Print to show the group and seperate list 


In [381]:

for group, cols in column_groups.items():
    print(f"\n{group.upper()}:\n", cols)


ID:
 ['candidate_id', 'resume_file_name', 'candidate_name']

PROFILE:
 ['primary_domain', 'primary_role', 'years_experience']

EDUCATION:
 ['highest_education', 'education_field', 'institution_tier']

SKILLS:
 ['technical_skills_raw', 'tools_platforms_raw', 'soft_skills_raw']

TEXT:
 ['experience_summary', 'project_summary', 'key_achievements']

FLAGS:
 ['management_experience_flag', 'people_management_flag', 'project_management_experience_flag', 'agile_scrum_experience_flag', 'client_facing_experience_flag', 'delivery_lead_experience_flag', 'cloud_experience_flag', 'ml_experience_flag', 'compliance_experience_flag', 'enterprise_systems_experience_flag', 'offshore_onsite_model_experience_flag', 'multi_vendor_coordination_flag', 'process_compliance_experience_flag', 'documentation_heavy_role_flag', 'mentoring_experience_flag', 'stakeholder_management_experience_flag']

SCORES:
 ['resume_length_words', 'keyword_density_score', 'profile_completeness_score']


In [382]:
important_flags = [
    'ml_experience_flag',
    'cloud_experience_flag',
    'project_management_experience_flag'
]

In [383]:
df['flag_score'] = df[important_flags].mean(axis=1)

# Score Normalization

In this step, we are converting different score values into a common scale between 0 and 1.

This process is called **Normalization**.

Machine Learning models work better when all feature values are in a similar range.

---

# 1. Keyword Score Normalization

```python
df['keyword_score_norm'] = df['keyword_density_score'] / 100
```

## What it does

`keyword_density_score` is originally in percentage format.

Example:

- 80%
- 65%
- 40%

We divide by 100 to convert it into values between 0 and 1.

### Example

```python
80 / 100 = 0.80
```

So:

- 80% → 0.80
- 50% → 0.50

---

# 2. Profile Completeness Score Normalization

```python
df['profile_score_norm'] = df['profile_completeness_score'] / 100
```

## What it does

Profile completeness is also a percentage value.

Example:

- 90%
- 70%
- 40%

We convert it into normalized decimal values.

### Example

```python
90 / 100 = 0.90
```

---

# 3. Resume Length Score Normalization

```python
df['length_score'] = df['resume_length_words'] / df['resume_length_words'].max()
```

## What it does

Different resumes have different word counts.

Example:

- Resume A = 500 words
- Resume B = 1000 words
- Resume C = 1500 words

We divide each resume length by the maximum resume length.

This converts all values between 0 and 1.

---

## Example

If maximum resume length is:

```python
1500
```

Then:

```python
500 / 1500 = 0.33
1000 / 1500 = 0.66
1500 / 1500 = 1.0
```

---

# Why We Do This

Normalization helps because:

- All features become comparable
- Large values do not dominate small values
- Machine Learning models perform better
- Training becomes more stable

---

# Simple Summary

We are converting all scores into a common scale (0 to 1)
so the Machine Learning model can understand and compare them properly.

In [384]:
df['keyword_score_norm'] = df['keyword_density_score'] / 100
df['profile_score_norm'] = df['profile_completeness_score'] / 100
df['length_score'] = df['resume_length_words'] / df['resume_length_words'].max()

## What we are going to do the next

### Resume data (skills, experience, etc.) -->Done

### Now we want to answer:

####  “Is this candidate good for a job?”

#### So we need to create simple scores.

## Computer understands lists, not messy text hence we create a list from technological raw data
### Create a method to retrun list and clean text with lower case and split where there is ',' 

In [385]:


def clean_skills(text):
    if pd.isnull(text):
        return []
    return text.lower().split(",")

# Now apply this to the technical_skills_raw to extract in the list and lower case


In [386]:

df['tech_skills_clean']=df['technical_skills_raw'].apply(clean_skills)


In [387]:
df[['technical_skills_raw', 'tech_skills_clean']].head()

,technical_skills_raw,tech_skills_clean
0,"Machine Learning, Power BI, Pandas, Python","[machine learning, power bi, pandas, python]"
1,"Kubernetes, Git, Docker, Python","[kubernetes, git, docker, python]"
2,"Due Diligence, Legal Research, Compliance, Con...","[due diligence, legal research, compliance, ..."
3,"Kubernetes, AWS, SQL, Docker","[kubernetes, aws, sql, docker]"
4,"Java, Kubernetes, Git, AWS","[java, kubernetes, git, aws]"


In [388]:
# ------------------------------------------------------------
# SKILL SCORE EXPLANATION
# ------------------------------------------------------------

# Every job requires different skills.
# So first we extract required skills from the Job Description (JD).

# Example:
# Data Scientist -> python, sql, machine learning
# Java Developer -> java, spring
# Finance Analyst -> excel, finance
# Lawyer -> legal research

# Example JD:
# "Looking for Python, SQL and AWS developer"

# Extracted JD skills:
# jd_skills = ['python', 'sql', 'aws']

# ------------------------------------------------------------
# RESUME SKILL EXTRACTION
# ------------------------------------------------------------

# Then we extract skills from the candidate resume.

# Example:
# candidate_skills = ['python', 'sql']

# ------------------------------------------------------------
# SKILL MATCH CALCULATION
# ------------------------------------------------------------

# Compare candidate skills with JD skills.

# JD Skills        = ['python', 'sql', 'aws']
# Candidate Skills = ['python', 'sql']

# Matched skills = 2
# Total JD skills = 3

# Skill Score = 2 / 3 = 0.66

# Final Skill Match = 66%

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------

# Higher score means:
# - Candidate skills match job requirements better

# Lower score means:
# - Candidate is missing important skills

In [389]:
def skill_match(candidate_skills, jd_skills):
    match = 0
    
    for skill in jd_skills:
        if skill in candidate_skills:
            match += 1
            
    return match / len(jd_skills)

In [390]:
jd_skills = ['python', 'sql', 'aws','machine learning','power bi', 'nlp', 'pandas']
df['skill_match_score'] = df['tech_skills_clean'].apply(
    lambda x: skill_match(x, jd_skills)
)

In [391]:
df[['tech_skills_clean', 'skill_match_score']].head(100)

,tech_skills_clean,skill_match_score
0,"[machine learning, power bi, pandas, python]",0.142857
1,"[kubernetes, git, docker, python]",0.000000
2,"[due diligence, legal research, compliance, ...",0.000000
3,"[kubernetes, aws, sql, docker]",0.000000
4,"[java, kubernetes, git, aws]",0.000000
...,...,...
95,"[payroll, hr analytics, employee engagement,...",0.000000
96,"[linux, sql, jaav, kubernetes]",0.000000
97,"[pandas, sql, power bi, python]",0.142857
98,"[sql, python, kubernetes, git]",0.142857


### We observed that there is spelling mispelt in many skill,  like pythono,pythno,jaav, we need to fix this
### how to fix it ??

--- Use Python built-in library get_close_matches

In [392]:
from difflib import get_close_matches
correct_skills = [
    "python", "sql", "java", "machine learning", 
    "aws", "excel", "power bi"
]
def fix_skill(skill):
    match = get_close_matches(skill, correct_skills, n=1, cutoff=0.6)
    
    if match:
        return match[0]
    else:
        return skill

In [393]:
#Apply to skill column 

def clean_and_fix_skills(skills):
    if pd.isnull(skills):
        return []
    
    skills_list = skills.lower().split(",")
    
    fixed = [fix_skill(skill.strip()) for skill in skills_list]
    
    return fixed

In [394]:
#Apply to dataframe
df['tech_skills_clean'] = df['technical_skills_raw'].apply(clean_and_fix_skills)

In [395]:
df[['tech_skills_clean', 'skill_match_score']].head(50)

,tech_skills_clean,skill_match_score
0,"[machine learning, power bi, pandas, python]",0.142857
1,"[kubernetes, git, docker, python]",0.000000
2,"[due diligence, legal research, compliance, co...",0.000000
3,"[kubernetes, aws, sql, docker]",0.000000
4,"[java, kubernetes, git, aws]",0.000000
5,"[nlp, machine learning, sql, python]",0.142857
6,"[kubernetes, aws, sql, java]",0.000000
7,"[kubernetes, aws, java, git]",0.000000
8,"[aws, kubernetes, java, python]",0.142857
9,"[machine learning, nlp, power bi, python]",0.142857


# Fuzzy matching techniques were used to correct spelling variations in skills (e.g., ‘ptyhon’ → ‘python’) to improve matching accuracy

## Experience Score

### What are we doing?

### We want to answer:

####  “Does candidate have enough experience?”
##### Assume job requirement 3 

In [396]:
required_exp = 3
df['exp_match_score'] = df['years_experience'] / required_exp

In [397]:
df['exp_match_score']

0       2.333333
1       3.333333
2       2.333333
3       3.000000
4       3.000000
          ...   
4995    2.000000
4996    3.333333
4997    0.333333
4998    2.666667
4999    1.000000
Name: exp_match_score, Length: 5000, dtype: float64

In [398]:
# limited to 1 year
df['exp_match_score'] = df['exp_match_score'].apply(lambda x: min(x, 1))


In [399]:
# Quick check 
df[['years_experience', 'exp_match_score']].head()

,years_experience,exp_match_score
0,7,1.0
1,10,1.0
2,7,1.0
3,9,1.0
4,9,1.0


## User Input: Experience

###  “The required experience parameter is designed to be dynamic and can be adjusted based on job requirements or user input via UI.

#### Note:
We will NOT hardcode values like "3 years" here.  
Instead, we are building a dynamic input system so that:
- The user can enter the required experience from the UI
- The model will automatically update the score based on the input

---

##  Example Inputs:

### • 2 years  
### • 5 years  

---

##  Model Behavior

#### Our model updates automatically

---

##  Streamlit Input Example

```python
required_exp = st.number_input(
    "Enter required experience",
    min_value=0,
    max_value=20,
    value=3
)

## 📊 Domain Score

 **Question:**  
“Is the candidate from the same field as the job?”

---

## 🎯 Example

### 💼 Job:
**Data Science**

---

### 👤 Candidate 1:
**Data Science**

👉 **Score = 1 ✅** (Perfect match)

---

### 👤 Candidate 2:
**Finance**

👉 **Score = 0 ❌** (No match)

In [400]:
domain_map = {
    "data science": ["data science", "data analyst", "machine learning", "ai"],
    "finance": ["finance", "accounting", "banking"],
    "law": ["law", "legal", "advocate"],
    "java developer": ["java", "spring", "backend"]
}

In [401]:
def domain_match_func(primary_domain, target_domain):
    primary_domain = str(primary_domain).lower()
    target_domain = target_domain.lower()

    if target_domain in domain_map:
        keywords = domain_map[target_domain]

        for k in keywords:
            if k in primary_domain:
                return 1

    return 0

In [402]:
#Create Domain Score
target_domain = "data science"

df['domain_match'] = df['primary_domain'].apply(
    lambda x: domain_match_func(x, target_domain)
)

In [403]:
# check the 
df[['primary_domain', 'domain_match']].head()

,primary_domain,domain_match
0,Data Science,1
1,IT,0
2,Legal,0
3,IT,0
4,IT,0


In [404]:
df.head()

,candidate_id,resume_file_name,candidate_name,primary_domain,primary_role,years_experience,highest_education,education_field,institution_tier,technical_skills_raw,...,keyword_density_score,profile_completeness_score,flag_score,keyword_score_norm,profile_score_norm,length_score,tech_skills_clean,skill_match_score,exp_match_score,domain_match
0,CND_100001,DataAnalyst_1.pdf,Candidate_1,Data Science,Data Analyst,7,LLM,Mathematics,Tier-1,"Machine Learning, Power BI, Pandas, Python",...,0.74,0.61,0.666667,0.0074,0.0061,0.888889,"[machine learning, power bi, pandas, python]",0.142857,1.0,1
1,CND_100002,profile_2.pdf,Candidate_2,IT,Senior Software Engineer,10,Masters,Computer Science,Tier-2,"Kubernetes, Git, Docker, Python",...,0.89,0.89,0.333333,0.0089,0.0089,0.857143,"[kubernetes, git, docker, python]",0.000000,1.0,0
2,CND_100003,Legal_candidate_3.pdf,Candidate_3,Legal,Compliance Officer,7,LLM,Law,Tier-1,"Due Diligence, Legal Research, Compliance, Con...",...,0.40,0.56,0.333333,0.0040,0.0056,0.904762,"[due diligence, legal research, compliance, co...",0.000000,1.0,0
3,CND_100004,IT_candidate_4.pdf,Candidate_4,IT,Senior Software Engineer,9,LLM,Computer Science,Tier-1,"Kubernetes, AWS, SQL, Docker",...,0.74,0.89,0.666667,0.0074,0.0089,0.873016,"[kubernetes, aws, sql, docker]",0.000000,1.0,0
4,CND_100005,IT_candidate_5.pdf,Candidate_5,IT,Software Engineer,9,Masters,Information Technology,Tier-1,"Java, Kubernetes, Git, AWS",...,0.58,0.59,0.666667,0.0058,0.0059,0.809524,"[java, kubernetes, git, aws]",0.000000,1.0,0


# NLP Applied 

## “How similar is resume text to job description?”
### Good Match: 
#### resume ="I worked on Python, machine learning, data analysis"
#### jd=We need Python and machine learning skills
#### These are similar → high score

### Bad match :

#### "Worked in finance and accounting"
#### Not similar → low score

In [405]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [406]:
def compute_similarity(resume, jd):
    
    # Convert 2 texts into numbers
    tfidf = TfidfVectorizer()
    vectors = tfidf.fit_transform([resume, jd])
    
    # Compare similarity
    score = cosine_similarity(vectors[0], vectors[1])
    
    return score[0][0]

In [407]:
def normalize_text(text):
    text = text.lower()
    text = text.replace("ml", "machine learning")
    text = text.replace("ai", "artificial intelligence")
    return text

In [408]:
resume = "Python machine learning experience"
jd = "Looking for Python, ML, SQL"

resume = normalize_text(resume)
jd = normalize_text(jd)

score = compute_similarity(jd, resume)
print(score )


0.4501755023269897


| Score Range | Meaning          |
| ----------- | ---------------- |
| 0.0 – 0.2   |  Very low match |
| 0.2 – 0.5   |  Partial match |
| 0.5 – 0.8   |  Good match     |
| 0.8 – 1.0   |  Strong match  |


In [409]:
#Apply in dataframe 

df['nlp_score'] = df.apply(
    lambda x: compute_similarity(
        str(x['experience_summary']),
        "Hands-on data science professional with 3+ year"
    ),
    axis=1
)

In [410]:
#check 

df[['experience_summary', 'nlp_score']].head(50)

,experience_summary,nlp_score
0,Hands-on data science professional with 7+ yea...,0.355934
1,"Over 10 years delivering solutions in it, coll...",0.047435
2,Hands-on legal professional with 7+ years of e...,0.212773
3,Hands-on it professional with 9+ years of end-...,0.212773
4,Hands-on it professional with 9+ years of end-...,0.212773
5,Hands-on data science professional with 2+ yea...,0.355934
6,"9+ years of experience in it roles, working ac...",0.000000
7,"Over 10 years delivering solutions in it, coll...",0.047435
8,"Over 5 years delivering solutions in it, colla...",0.048851
9,Hands-on data science professional with 5+ yea...,0.355934


## We found that education filed also add weightage to the final score hence we add the education field 


In [411]:

valid_education = [
    "data science",
    "computer science",
    "information technology",
    "artificial intelligence",
    "machine learning"
]

In [412]:
def check_education(edu):
    if pd.isnull(edu):
        return 0
    
    edu = edu.lower()
    
    # Full forms (normal match)
    full_forms = [
        "data science",
        "computer science",
        "information technology",
        "artificial intelligence",
        "machine learning"
    ]
    
    for f in full_forms:
        if f in edu:
            return 1
    
    # Short forms (exact match)
    short_forms = ["it", "ai", "cs","ds"]
    
    for s in short_forms:
        if edu.strip() == s:
            return 1
    
    return 0

In [413]:
df['education_match'] = df['education_field'].apply(check_education)

In [414]:
df[['education_field', 'education_match']].head(50)

,education_field,education_match
0,Mathematics,0
1,Computer Science,1
2,Law,0
3,Computer Science,1
4,Information Technology,1
5,Mathematics,0
6,Computer Science,1
7,Information Technology,1
8,Information Technology,1
9,Mathematics,0


In [415]:
df['final_score'] = (
    0.25 * df['skill_match_score'] +
    0.25 * df['exp_match_score'] +  
    0.15 * df['domain_match'] +
    0.15 * df['education_match'] +
    0.1 * df['flag_score'] +
    0.05 * df['keyword_score_norm'] +
    0.05 * df['profile_score_norm']
)


## skill + exp + domain + education + flags → final_score → selected

In [416]:
df[['skill_match_score', 'exp_match_score', 'domain_match','education_match','flag_score','final_score']].head(50)

,skill_match_score,exp_match_score,domain_match,education_match,flag_score,final_score
0,0.142857,1.000000,1,0,0.666667,0.503056
1,0.000000,1.000000,0,1,0.333333,0.434223
2,0.000000,1.000000,0,0,0.333333,0.283813
3,0.000000,1.000000,0,1,0.666667,0.467482
4,0.000000,1.000000,0,1,0.666667,0.467252
5,0.142857,0.666667,1,0,0.333333,0.386519
6,0.000000,1.000000,0,1,0.666667,0.467362
7,0.000000,1.000000,0,1,0.666667,0.467187
8,0.142857,1.000000,0,1,0.666667,0.503016
9,0.142857,1.000000,1,0,0.666667,0.502811


## Apply model
### If score > 0.6 → good candidate

In [417]:

df['selected'] = df['final_score'].apply(lambda x: 1 if x > 0.60 else 0) 

In [418]:
df.head(50)

,candidate_id,resume_file_name,candidate_name,primary_domain,primary_role,years_experience,highest_education,education_field,institution_tier,technical_skills_raw,...,profile_score_norm,length_score,tech_skills_clean,skill_match_score,exp_match_score,domain_match,nlp_score,education_match,final_score,selected
0,CND_100001,DataAnalyst_1.pdf,Candidate_1,Data Science,Data Analyst,7,LLM,Mathematics,Tier-1,"Machine Learning, Power BI, Pandas, Python",...,0.0061,0.888889,"[machine learning, power bi, pandas, python]",0.142857,1.000000,1,0.355934,0,0.503056,0
1,CND_100002,profile_2.pdf,Candidate_2,IT,Senior Software Engineer,10,Masters,Computer Science,Tier-2,"Kubernetes, Git, Docker, Python",...,0.0089,0.857143,"[kubernetes, git, docker, python]",0.000000,1.000000,0,0.047435,1,0.434223,0
2,CND_100003,Legal_candidate_3.pdf,Candidate_3,Legal,Compliance Officer,7,LLM,Law,Tier-1,"Due Diligence, Legal Research, Compliance, Con...",...,0.0056,0.904762,"[due diligence, legal research, compliance, co...",0.000000,1.000000,0,0.212773,0,0.283813,0
3,CND_100004,IT_candidate_4.pdf,Candidate_4,IT,Senior Software Engineer,9,LLM,Computer Science,Tier-1,"Kubernetes, AWS, SQL, Docker",...,0.0089,0.873016,"[kubernetes, aws, sql, docker]",0.000000,1.000000,0,0.212773,1,0.467482,0
4,CND_100005,IT_candidate_5.pdf,Candidate_5,IT,Software Engineer,9,Masters,Information Technology,Tier-1,"Java, Kubernetes, Git, AWS",...,0.0059,0.809524,"[java, kubernetes, git, aws]",0.000000,1.000000,0,0.212773,1,0.467252,0
5,CND_100006,Data Science_candidate_6.pdf,Candidate_6,Data Science,Junior Data Scientist,2,Bachelors,Mathematics,Tier-1,"NLP, Machine Learning, SQL, Python",...,0.0093,0.857143,"[nlp, machine learning, sql, python]",0.142857,0.666667,1,0.355934,0,0.386519,0
6,CND_100007,DevOpsEngineer_7.pdf,Candidate_7,IT,DevOps Engineer,9,NaN,Computer Science,Tier-1,"Kubernetes, AWS, SQL, Java",...,0.0058,0.873016,"[kubernetes, aws, sql, java]",0.000000,1.000000,0,0.000000,1,0.467362,0
7,CND_100008,resume_8_Full_Stack_Developer.pdf,Candidate_8,IT,Full Stack Developer,10,Bachelors,Information Technology,Tier-1,"Kubernetes, AWS, Java, Git",...,0.0063,0.809524,"[kubernetes, aws, java, git]",0.000000,1.000000,0,0.047435,1,0.467187,0
8,CND_100009,resume_9_Cloud_Engineer.pdf,Candidate_9,IT,Cloud Engineer,5,Masters,Information Technology,Tier-2,"AWS, Kubernetes, Java, Ptyhon",...,0.0083,0.825397,"[aws, kubernetes, java, python]",0.142857,1.000000,0,0.048851,1,0.503016,0
9,CND_100010,resume_10_ML_Engineer.pdf,Candidate_10,Data Science,ML Engineer,5,Masters,Mathematics,Tier-3,"Machine Learning, NLP, Power BI, Pythno",...,0.0058,0.888889,"[machine learning, nlp, power bi, python]",0.142857,1.000000,1,0.355934,0,0.502811,0


In [419]:
df[['final_score', 'selected']].head(50)

,final_score,selected
0,0.503056,0
1,0.434223,0
2,0.283813,0
3,0.467482,0
4,0.467252,0
5,0.386519,0
6,0.467362,0
7,0.467187,0
8,0.503016,0
9,0.502811,0


In [420]:
X = df[
    ['skill_match_score', 'exp_match_score', 'domain_match',
     'education_match', 'flag_score',
     'keyword_score_norm', 'profile_score_norm', 'length_score']
]

y = df['selected']

In [421]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [422]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [423]:
y_pred = model.predict(X_test)

In [424]:
#accuracy 

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


In [425]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

domain_match          0.361013
education_match       0.360796
length_score          0.053389
skill_match_score     0.049036
keyword_score_norm    0.048928
exp_match_score       0.048127
flag_score            0.041128
profile_score_norm    0.037583
dtype: float64


In [426]:
# Save model
import pickle
with open("../models/model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


In [427]:
df['selected'].value_counts()

selected
0    4727
1     273
Name: count, dtype: int64

# Observed Biased in the Model
### We observed the selected count is very low i.e 273 and it biased 
#### Hence we will include top %
#### top_percent = 0.3   # means top 30%

In [428]:
top_percent = 0.3 
cutoff = df['final_score'].quantile(1 - top_percent)
df['selected'] = df['final_score'].apply(lambda x: 1 if x >= cutoff else 0)

In [429]:
df['selected'].value_counts()

selected
0    3498
1    1502
Name: count, dtype: int64

# We observed the selected count i.e 1502 and it not biased 
## Hence now we can go with train and split

In [430]:
X = df[
    ['skill_match_score', 'exp_match_score', 'domain_match',
     'education_match', 'flag_score',
     'keyword_score_norm', 'profile_score_norm', 'length_score']
]

y = df['selected']

In [431]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Why We Use RandomForestClassifier

RandomForestClassifier is a Machine Learning algorithm used for classification problems.

In this project, we want to predict:

- Selected = 1
- Rejected = 0

So this becomes a classification problem.

---

# Why Random Forest?

## 1. Works well with numeric features

Our model uses numeric resume scores like:

- Skill score
- Experience score
- Domain score
- Education score

Random Forest works very well with these types of structured numeric features.

---

## 2. Gives good accuracy

Random Forest usually gives good prediction accuracy without requiring heavy tuning.

---

## 3. Handles multiple factors together

Resume screening depends on many factors:

- Skills
- Experience
- Education
- Domain match
- Keywords

Random Forest can understand the relationship between all these features together.

---

## 4. Reduces overfitting

Random Forest creates multiple Decision Trees instead of one tree.

Each tree gives its own prediction.

Example:

- Tree 1 → Selected
- Tree 2 → Selected
- Tree 3 → Rejected

Final prediction is based on majority voting.

This helps reduce overfitting and improves stability.

---

## 5. Easy to train and use

Random Forest is:
- Simple to implement
- Fast to train
- Good for beginner and intermediate ML projects

---

# Why It Is Good for Resume Screening

Resume screening contains many scoring parameters.

Random Forest can analyze:
- candidate skills
- experience
- education
- profile quality

and make a final prediction based on all these factors.

---

# Simple Summary

We use RandomForestClassifier because:

- It is good for classification problems
- Works well with resume scoring features
- Gives good accuracy
- Handles multiple inputs together
- Reduces overfitting
- Easy to implement in real projects

In [432]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [433]:
y_pred = model.predict(X_test)

## Accuracy Score


In [434]:

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.999


In [435]:
print(len(X_train))
print(len(y_train))
print(type(X_train))
print(type(y_train))

4000
4000
<class 'pandas.DataFrame'>
<class 'pandas.Series'>


In [436]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

skill_match_score     0.560948
domain_match          0.151384
flag_score            0.116508
exp_match_score       0.053402
education_match       0.046776
keyword_score_norm    0.028412
profile_score_norm    0.024181
length_score          0.018389
dtype: float64


# Save model


In [437]:
import pickle
with open("../models/model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


## Accuracy Report

In [438]:
preds = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, preds))
print("\\nClassification Report:\\n")
print(classification_report(y_test, preds))

Accuracy: 0.999
\nClassification Report:\n
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       709
           1       1.00      1.00      1.00       291

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

